# BLIP-2 leave-one-query-out ablation (LOQOA)

**Research question:** Which of BLIP-2's 32 pretrained Q-Former queries have the largest marginal contribution to zero-shot VQAv2 accuracy when all other queries are retained?

This is an inference-only experiment using `Salesforce/blip2-flan-t5-xl`. No training or new query embeddings are introduced. The reference uses all 32 queries; each of the 32 ablations removes exactly one original embedding *before* Q-Former processing, leaving 31 in their original order. Query IDs are zero-based (0–31). All conditions use the same images, questions, prompt and decoding settings.

For query $i$, the eventual importance measure is $I_i=A_{32}-A_{-i}$, where accuracy is official VQAv2 accuracy in percentage points. Positive values indicate that removal lowered accuracy. This is conditional marginal importance, not a measure of independent or universal query usefulness: queries can interact or be redundant. The smoke stage is only a functionality check.

Run in the project's BLIP-2 CUDA or ROCm Jupyter kernel, from the repository root or `scripts/`. Reuses `load_blip2.ipynb` and the existing `data/validation_dataset/` subset. Save this notebook as **`scripts/leave_one_query_out.ipynb`** before running. Do not run another GPU experiment simultaneously. Predictions go to `results/leave_one_query_out/`, separate from the query-count experiment. The current `score_results.ipynb` is for the original query-count experiment; LOQOA scoring will require a separate extension using the official evaluator.

## 1. Select the run size and plan compute

Change **only** `NUM_QUESTIONS` in the next cell, save the notebook and rerun from the beginning. The default is 10 questions **per condition**, not 10 predictions total. All 33 conditions run sequentially, batch size 1, with two unmeasured warmups per unfinished condition. The selected questions are a reproducible random permutation of the existing 2,000-question subset (seed 42), with nested prefixes. Each stage has its own results directory; rerunning the same stage resumes unfinished predictions. Larger stages do not automatically reuse smaller-stage predictions.

| `NUM_QUESTIONS` | Stage | Scored predictions | Warmup generations (fresh run) |
|---:|---|---:|---:|
| 10 | smoke | 330 | 66 |
| 200 | pilot | 6,600 | 66 |
| 2000 | main | 66,000 | 66 |

**Runtime estimation:** Let $T_{10}$ and $T_{200}$ be measured elapsed seconds for fresh, complete runs of the entire inference loop (all 33 conditions, including warmups, preprocessing and saving; excluding model loading and manifest setup). A simple proportional estimate from 10 questions is $200T_{10}$; from 200 questions it is $10T_{200}$. The 10-question estimate can be inflated by fixed overhead. When both timings are available, estimate a fixed-overhead plus linear-cost model:

$$b=\frac{T_{200}-T_{10}}{190},\qquad a=T_{10}-10b,\qquad \widehat T_{2000}=a+2000b=T_{200}+1800b.$$

Here $b$ is seconds per additional question **across all 33 conditions**. Divide seconds by 3600 to obtain hours. For example, if the complete 10-question loop takes 120 seconds and the 200-question loop takes 1,440 seconds, the two-point estimate is about 3.87 hours. These are illustrative numbers, not measured results. Use measurements from the same machine, checkpoint, code and decoding settings. If the fitted overhead or slope is negative, prefer the 200-question proportional estimate. The final cell calculates projections automatically when compatible completed stages are available. Cold model loading, thermal throttling, other GPU processes, and interruptions can change the actual runtime; a resumed run's current-session wall time is not a valid full-run timing.

In [ ]:
NUM_QUESTIONS = 10  # 10 = smoke, 200 = pilot, 2000 = main

## 2. Shared settings and utilities

The prompt and generation settings match the query-count ablation. The hardware backend is recorded automatically; CUDA and ROCm results are not mixed.

In [ ]:
import os
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")  # Preserve an existing user override.

from pathlib import Path
import hashlib
import json
import platform
import random
import time
import uuid
import math
import weakref
from datetime import datetime, timezone
from importlib.metadata import version

import nbformat
import torch
from filelock import FileLock
from PIL import Image
from tqdm.auto import tqdm

STAGES = {10: "smoke", 200: "pilot", 2000: "main"}
if NUM_QUESTIONS not in STAGES:
    raise ValueError("NUM_QUESTIONS must be 10, 200, or 2000.")
STAGE = STAGES[NUM_QUESTIONS]
QUESTION_ORDER_SEED = 42
RUN_ORDER_SEED = 2026
WARMUP_QUESTIONS = 2
PIPELINE_VERSION = 1
PROMPT = "Question: {question} Short answer:"
GENERATION_SETTINGS = {
    "do_sample": False,
    "num_beams": 5,
    "length_penalty": -1.0,
    "max_new_tokens": 10,
}

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA-enabled or ROCm-enabled PyTorch environment is required.")
GPU_BACKEND = "rocm" if torch.version.hip else "cuda" if torch.version.cuda else None
if GPU_BACKEND is None:
    raise RuntimeError("Could not identify a CUDA or ROCm PyTorch build.")


cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "scripts" else cwd
DATA_DIR = PROJECT_ROOT / "data" / "validation_dataset"
IMAGE_DIR = DATA_DIR / "images" / "val2014"
RESULTS_ROOT = PROJECT_ROOT / "results" / "leave_one_query_out"


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".partial")
    try:
        with temporary.open("w", encoding="utf-8") as stream:
            json.dump(value, stream, indent=2, allow_nan=False)
            stream.write("\n")
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def image_path(question):
    return IMAGE_DIR / f"COCO_val2014_{int(question['image_id']):012d}.jpg"


## 3. Load the nested validation sample

Retain all ten human annotations for subsequent official scoring. Ground truth is never passed to the model.

In [ ]:
questions_path = DATA_DIR / "v2_OpenEnded_mscoco_val2014_questions_subset.json"
annotations_path = DATA_DIR / "v2_mscoco_val2014_annotations_subset.json"
all_questions = json.loads(questions_path.read_text(encoding="utf-8"))["questions"]
annotations = json.loads(annotations_path.read_text(encoding="utf-8"))["annotations"]
annotation_by_id = {int(item["question_id"]): item for item in annotations}
if len(annotation_by_id) != len(annotations):
    raise ValueError("Duplicate annotation question IDs.")
if len(all_questions) != 2000:
    raise ValueError("Expected the existing 2,000-question validation subset.")
ordered_questions = list(all_questions)
random.Random(QUESTION_ORDER_SEED).shuffle(ordered_questions)
questions = ordered_questions[:NUM_QUESTIONS]
question_ids = [int(question["question_id"]) for question in questions]
if len(set(question_ids)) != len(question_ids):
    raise ValueError("Duplicate question IDs in the selected data.")
for question in questions:
    annotation = annotation_by_id[int(question["question_id"])]
    if int(annotation["image_id"]) != int(question["image_id"]):
        raise ValueError("Question and annotation image IDs differ.")
    if len(annotation["answers"]) != 10:
        raise ValueError("Official VQA scoring requires all ten annotator answers.")
    if not image_path(question).is_file():
        raise FileNotFoundError(image_path(question))

image_hashes = {path.name: file_sha256(path) for path in sorted({image_path(q) for q in questions})}
print(f"Stage {STAGE}: {len(questions)} questions, {len(image_hashes)} distinct images.")


## 4. Reuse BLIP-2 and define all 33 conditions

The existing notebook loader reuses the model if already present in the same kernel. The original K=32 parameter is preserved; each ablation removes exactly one query from that original set. Restore the reference after every condition and after interruptions. A single resident GPU and BF16 are required, matching the main experiment. The baseline is run first; the omitted-query order is reproducibly shuffled.

In [ ]:
# Restore only a query parameter that belongs to this exact model instance.
# A fresh/replaced model must never receive a parameter from a previous model.
if "_loqo_model_ref" in globals() and _loqo_model_ref() is globals().get("model") and globals().get("model") is not None:
    restore_loqo_original_queries()

loader = PROJECT_ROOT / "scripts" / "load_blip2.ipynb"
get_ipython().run_line_magic("run", f'-i "{loader.as_posix()}"')

if tuple(model.query_tokens.shape) != (1, 32, 768) or model.config.num_query_tokens != 32 or processor.num_query_tokens != 32:
    raise RuntimeError("Start from the original 32-query checkpoint in a fresh kernel.")
if processor.tokenizer.convert_tokens_to_ids("<image>") != model.config.image_token_index:
    raise RuntimeError("The processor and model image-token IDs must match.")

vision_parameter = next(model.vision_model.parameters())
DEVICE = vision_parameter.device
DTYPE = vision_parameter.dtype
if DEVICE.type != "cuda":
    raise RuntimeError("The vision encoder must be on the GPU.")
with torch.cuda.device(DEVICE):
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError("The selected GPU must support BF16 inference.")
if DTYPE != torch.bfloat16:
    raise RuntimeError(f"Expected BF16 vision inputs, got {DTYPE}.")
# A single resident GPU is required for comparable timing/memory metrics and
# safe query replacement. Do not silently benchmark CPU/disk offloading.
parameter_devices = {parameter.device for parameter in model.parameters()}
if parameter_devices != {DEVICE}:
    raise RuntimeError(
        f"Expected the whole model on {DEVICE}; found {parameter_devices}. "
        "Stop other GPU kernels and check the shared loader's placement before proceeding."
    )

model.requires_grad_(False)
model.eval()
_loqo_model_ref = weakref.ref(model)
loqo_original_parameter = model.query_tokens
loqo_original_queries = loqo_original_parameter.detach().clone()


def loqo_query_indices(removed_query):
    """Return the retained original indices, in order."""
    if removed_query is None:
        return torch.arange(32, device="cpu")
    if not isinstance(removed_query, int) or not 0 <= removed_query < 32:
        raise ValueError("Removed query must be an integer from 0 to 31 or None.")
    return torch.tensor([i for i in range(32) if i != removed_query], dtype=torch.long)


def restore_loqo_original_queries():
    """Restore the original parameter, configuration and processor; verify values."""
    if _loqo_model_ref() is not model:
        raise RuntimeError("Model instance changed; restart from the pretrained checkpoint.")
    model.query_tokens = loqo_original_parameter
    model.config.num_query_tokens = 32
    processor.num_query_tokens = 32
    if (tuple(model.query_tokens.shape) != (1, 32, 768)
            or model.config.num_query_tokens != 32
            or processor.num_query_tokens != 32
            or not torch.equal(model.query_tokens.detach(), loqo_original_queries)):
        raise RuntimeError("Original query restoration failed. Restart the kernel.")


def set_removed_query(removed_query):
    """Always select from the untouched original 32-query parameter."""
    if _loqo_model_ref() is not model:
        raise RuntimeError("Model instance changed; rerun the model setup cell.")
    indices = loqo_query_indices(removed_query)
    k = len(indices)
    if removed_query is None:
        restore_loqo_original_queries()
    else:
        selected = loqo_original_queries.index_select(1, indices.to(loqo_original_queries.device))
        model.query_tokens = torch.nn.Parameter(selected, requires_grad=False)
        model.config.num_query_tokens = k
        processor.num_query_tokens = k
    if tuple(model.query_tokens.shape) != (1, k, 768) or model.config.num_query_tokens != k or processor.num_query_tokens != k:
        raise RuntimeError("Query count and processor configuration are inconsistent.")
    return indices.tolist()


conditions = [{"name": "k032", "k": 32, "seed": None, "removed_query": None,
               "query_indices": list(range(32)), "kind": "baseline"}]
removed_queries = list(range(32))
random.Random(RUN_ORDER_SEED).shuffle(removed_queries)
conditions += [
    {"name": f"drop-query-{i:02d}", "k": 31, "seed": None, "removed_query": i,
     "query_indices": loqo_query_indices(i).tolist(), "kind": "leave-one-query-out"}
    for i in removed_queries
]
for condition in conditions:
    assert len(condition["query_indices"]) == condition["k"]
    if condition["removed_query"] is not None:
        assert set(range(32)) - set(condition["query_indices"]) == {condition["removed_query"]}
assert len(conditions) == 33
print(f"{GPU_BACKEND.upper()}: {torch.cuda.get_device_name(DEVICE)} | {DTYPE} | {len(conditions)} conditions")


## 5. Record the experiment manifest

The signature records the checkpoint, code, retained query indices, sample IDs, image hashes, generation settings and hardware. The tagged run-size cell is excluded from the code hash so that changing only the number of questions permits comparison of timing pilots; the stage and selected IDs remain part of the experiment signature. Separate stages and hardware produce separate directories.

In [ ]:
checkpoint_revision = getattr(model.config, "_commit_hash", None)
if not checkpoint_revision:
    raise RuntimeError("Could not identify the loaded Hugging Face checkpoint revision.")
notebook = nbformat.read(PROJECT_ROOT / "scripts" / "leave_one_query_out.ipynb", as_version=4)
code_digest = hashlib.sha256("\n".join(cell.source for cell in notebook.cells if cell.cell_type == "code" and "runtime-config" not in cell.metadata.get("tags", [])).encode()).hexdigest()
packages = ["torch", "torchvision", "transformers", "accelerate", "huggingface_hub", "numpy", "Pillow", "nbformat", "filelock", "tqdm"]
experiment = {
    "pipeline_version": PIPELINE_VERSION,
    "notebook_code_sha256": code_digest,
    "loader_sha256": file_sha256(loader),
    "stage": STAGE,
    "model_id": MODEL_ID,
    "checkpoint_revision": checkpoint_revision,
    "original_query_sha256": hashlib.sha256(loqo_original_queries.cpu().contiguous().view(torch.uint8).numpy().tobytes()).hexdigest(),
    "question_file_sha256": file_sha256(questions_path),
    "annotation_file_sha256": file_sha256(annotations_path),
    "question_ids": question_ids,
    "image_sha256": image_hashes,
    "prompt": PROMPT,
    "generation": GENERATION_SETTINGS,
    "batch_size": 1,
    "warmup_questions": WARMUP_QUESTIONS,
    "precision": str(DTYPE),
    "device_map": {key: str(value) for key, value in model.hf_device_map.items()},
    "gpu_name": torch.cuda.get_device_name(DEVICE),
    "gpu_memory_bytes": torch.cuda.get_device_properties(DEVICE).total_memory,
    "gpu_backend": GPU_BACKEND,
    "cuda_version": torch.version.cuda,
    "rocm_version": torch.version.hip,
    "gpu_device_index": DEVICE.index,
    "query_dtype": str(loqo_original_queries.dtype),
    "parameter_dtypes": sorted({str(p.dtype) for p in model.parameters()}),
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "packages": {name: version(name) for name in packages},
    "run_order_seed": RUN_ORDER_SEED,
    "question_order_seed": QUESTION_ORDER_SEED,
    "experiment_type": "leave_one_query_out",
    "conditions": conditions,
}
# The runtime comparison key excludes only the stage and selected sample fields.
# All other settings (including dataset hashes, hardware, code and conditions) must match.
comparison_settings = {key: value for key, value in experiment.items()
                       if key not in {"stage", "question_ids", "image_sha256"}}
runtime_key = hashlib.sha256(json.dumps(comparison_settings, sort_keys=True).encode()).hexdigest()
experiment["runtime_comparison_key"] = runtime_key
signature = hashlib.sha256(json.dumps(experiment, sort_keys=True).encode()).hexdigest()
EXPERIMENT_DIR = RESULTS_ROOT / f"{STAGE}-{signature[:16]}"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
print("Results directory:", EXPERIMENT_DIR)
print(f"Planned predictions: {len(conditions) * len(questions):,}")


## 6. Run or resume inference

Each condition writes `records.jsonl` incrementally, official-format `predictions.json`, and `status.json`. The lock prevents concurrent writers; interrupted final JSONL writes are recovered. On rerun, validated records are reused. Timing covers synchronized generation and per-question preprocessing/decoding, plus a separate full-loop wall measurement for a fresh completed run. No training or optimizer is used.

In [ ]:
def read_records(path, expected_ids, condition):
    """Recover an interrupted final write and validate the saved ordered prefix."""
    if not path.exists():
        return []
    records = []
    with path.open("rb+") as stream:
        while True:
            start = stream.tell()
            line = stream.readline()
            if not line:
                break
            if not line.endswith(b"\n"):
                stream.truncate(start)
                stream.flush()
                os.fsync(stream.fileno())
                break
            record = json.loads(line)
            position = len(records)
            if position >= len(expected_ids) or record["question_id"] != expected_ids[position]:
                raise ValueError(f"Unexpected or duplicate question in {path}.")
            if record["k"] != condition["k"] or record["removed_query"] != condition["removed_query"]:
                raise ValueError(f"Saved condition differs in {path}.")
            if record["image_id"] != int(annotation_by_id[record["question_id"]]["image_id"]):
                raise ValueError(f"Saved image ID differs in {path}.")
            if (not isinstance(record["answer"], str)
                    or not math.isfinite(record["generation_seconds"])
                    or record["generation_seconds"] <= 0):
                raise ValueError(f"Invalid prediction record in {path}.")
            records.append(record)
    return records


def generate_answer(question):
    question_started = time.perf_counter()
    with Image.open(image_path(question)) as image:
        inputs = processor(
            images=image.convert("RGB"),
            text=PROMPT.format(question=question["question"]),
            return_tensors="pt",
        )
    placeholder_count = int((inputs["input_ids"] == model.config.image_token_index).sum())
    if placeholder_count != model.query_tokens.shape[1]:
        raise RuntimeError("Image placeholder count differs from the current query count.")
    inputs = inputs.to(device=DEVICE, dtype=DTYPE)
    torch.cuda.synchronize(DEVICE)
    torch.cuda.reset_peak_memory_stats(DEVICE)
    allocated_before = torch.cuda.memory_allocated(DEVICE)
    started = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(**inputs, **GENERATION_SETTINGS)
    torch.cuda.synchronize(DEVICE)
    elapsed = time.perf_counter() - started
    peak_allocated = torch.cuda.max_memory_allocated(DEVICE)
    peak_reserved = torch.cuda.max_memory_reserved(DEVICE)
    tokens = generated[0, 1:]  # Flan-T5 output begins with a decoder start token.
    length = int((tokens != processor.tokenizer.pad_token_id).sum())
    answer = processor.batch_decode(generated, skip_special_tokens=True)[0].strip()
    return {
        "question_id": int(question["question_id"]),
        "image_id": int(question["image_id"]),
        "answer": answer,
        "generation_seconds": elapsed,
        "question_seconds": time.perf_counter() - question_started,
        "generated_token_count": length,
        "allocated_before_bytes": allocated_before,
        "peak_allocated_bytes": peak_allocated,
        "peak_reserved_bytes": peak_reserved,
    }


def run_condition(condition):
    directory = EXPERIMENT_DIR / condition["name"]
    directory.mkdir(exist_ok=True)
    path = directory / "records.jsonl"
    records = read_records(path, question_ids, condition)
    remaining = questions[len(records):]
    try:
        if remaining:
            indices = set_removed_query(condition["removed_query"])
            if indices != condition["query_indices"]:
                raise RuntimeError("Active query indices differ from the manifest.")
            torch.cuda.empty_cache()
            for question in questions[:WARMUP_QUESTIONS]:
                generate_answer(question)
            write_json_atomic(directory / "status.json", {"state": "running", "completed": len(records)})
            with path.open("a", encoding="utf-8") as stream:
                for question in tqdm(remaining, desc=condition["name"]):
                    record = generate_answer(question)
                    record.update(k=condition["k"], query_seed=None, removed_query=condition["removed_query"])
                    stream.write(json.dumps(record, allow_nan=False) + "\n")
                    stream.flush()
                    os.fsync(stream.fileno())
                    records.append(record)
        write_json_atomic(directory / "predictions.json", [
            {"question_id": record["question_id"], "answer": record["answer"]} for record in records
        ])
        write_json_atomic(directory / "status.json", {"state": "complete", "completed": len(records)})
        return records
    except BaseException as error:
        write_json_atomic(directory / "status.json", {
            "state": "interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
            "completed": len(records), "error": type(error).__name__ + ": " + str(error),
        })
        raise
    finally:
        restore_loqo_original_queries()


# A single writer owns this experiment directory, including recovery and exports.
with FileLock(str(EXPERIMENT_DIR / "experiment.lock"), timeout=0):
    manifest_path = EXPERIMENT_DIR / "manifest.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest["signature"] != signature or manifest["experiment"] != experiment:
            raise ValueError("The saved manifest differs from this experiment.")
    else:
        write_json_atomic(manifest_path, {
            "signature": signature, "created_utc": datetime.now(timezone.utc).isoformat(),
            "experiment": experiment,
        })
    complete_path = EXPERIMENT_DIR / "complete.json"
    timing_path = EXPERIMENT_DIR / "runtime.json"
    previously_complete = complete_path.exists()
    fresh_run = not any((EXPERIMENT_DIR / condition["name"] / "records.jsonl").exists()
                        for condition in conditions)
    complete_path.unlink(missing_ok=True)
    all_records = []
    started = time.perf_counter()
    try:
        for condition in experiment["conditions"]:
            all_records.extend(run_condition(condition))
        loop_elapsed = time.perf_counter() - started
        if len(all_records) != len(conditions) * len(questions):
            raise RuntimeError("Incomplete experiment.")
        timing = {
            "signature": signature,
            "runtime_comparison_key": runtime_key,
            "questions_per_condition": len(questions),
            "condition_count": len(conditions),
            "prediction_count": len(all_records),
            "generation_seconds_total": sum(r["generation_seconds"] for r in all_records),
            "question_seconds_total": sum(r["question_seconds"] for r in all_records),
            "full_fresh_loop_seconds": loop_elapsed if fresh_run else None,
            "last_session_loop_seconds": loop_elapsed,
        }
        # A no-op rerun must not replace a valid full-run wall measurement.
        if previously_complete and timing_path.exists():
            previous = json.loads(timing_path.read_text(encoding="utf-8"))
            if previous["signature"] == signature:
                timing["full_fresh_loop_seconds"] = previous["full_fresh_loop_seconds"]
        write_json_atomic(timing_path, timing)
        write_json_atomic(complete_path, {
            "signature": signature, "condition_count": len(conditions),
            "questions_per_condition": len(questions),
            "finished_utc": datetime.now(timezone.utc).isoformat(),
        })
    finally:
        restore_loqo_original_queries()
        torch.cuda.empty_cache()

print(f"Finished {STAGE}: {len(conditions)} conditions, {len(all_records):,} predictions. Original K=32 restored.")
print("Results:", EXPERIMENT_DIR)


## 7. Runtime estimates from completed runs

This CPU-only summary finds completed smoke/pilot/main runs with the same runtime comparison key. The full-loop projection uses only fresh-run wall times. If a run was resumed, summed per-question timings can still provide a lower-overhead approximate estimate, but exclude warmups and disk writes. The 200-question pilot is generally more informative than the smoke test. Model loading time is excluded from all projections.

In [ ]:
import pandas as pd

runtime_rows = []
for directory in sorted(RESULTS_ROOT.glob("*-*")):
    if not (directory / "complete.json").is_file() or not (directory / "runtime.json").is_file():
        continue
    row = json.loads((directory / "runtime.json").read_text(encoding="utf-8"))
    completion = json.loads((directory / "complete.json").read_text(encoding="utf-8"))
    if row["runtime_comparison_key"] != runtime_key or row["signature"] != completion["signature"]:
        continue
    runtime_rows.append({**row, "directory": directory.name})

runtime_table = pd.DataFrame(runtime_rows).sort_values("questions_per_condition")
columns = ["questions_per_condition", "prediction_count", "generation_seconds_total",
           "question_seconds_total", "full_fresh_loop_seconds", "directory"]
display(runtime_table[columns].round(3))

estimates = []
for row in runtime_rows:
    count = row["questions_per_condition"]
    if count not in (10, 200):
        continue
    wall = row["full_fresh_loop_seconds"]
    estimates.append({"method": f"{count}-question per-question extrapolation (excludes warmup/writes)",
                      "estimated_hours": row["question_seconds_total"] * 2000 / count / 3600})
    if wall is not None:
        estimates.append({"method": f"{count}-question full-loop proportional extrapolation",
                          "estimated_hours": wall * 2000 / count / 3600})

fresh = {count: [r["full_fresh_loop_seconds"] for r in runtime_rows
                 if r["questions_per_condition"] == count and r["full_fresh_loop_seconds"] is not None]
         for count in (10, 200)}
if len(fresh[10]) == 1 and len(fresh[200]) == 1:
    t10, t200 = fresh[10][0], fresh[200][0]
    slope = (t200 - t10) / 190
    overhead = t10 - 10 * slope
    if slope > 0 and overhead >= 0:
        estimates.append({"method": "Two-point full-loop estimate (10 and 200)",
                          "estimated_hours": (t200 + 1800 * slope) / 3600})
    else:
        print("Two-point fit has a nonphysical slope or overhead; prefer the 200-question proportional estimate.")
if estimates:
    estimate_table = pd.DataFrame(estimates)
    display(estimate_table.round(3))
    estimate_table.to_csv(EXPERIMENT_DIR / "runtime_estimates.csv", index=False)
else:
    print("Complete a 10- or 200-question stage to obtain a runtime estimate.")


## 8. Next steps: official scoring

- Smoke: verify all 33 conditions contain 10 predictions (330 total) and that the original K=32 parameter was restored.
- Pilot: use 200 questions to inspect runtime and exploratory query importance. Main: use all 2,000 for stronger inference.
- Use the official VQAv2 evaluator to score all conditions against the same annotations. Compute paired accuracy differences $I_i=A_{32}-A_{-i}$ and image-cluster bootstrap intervals. Query importance comparisons across 32 omissions are exploratory unless multiplicity is addressed. Do not select the best subset on these data and claim independent validation performance.
- The existing `score_results.ipynb` is designed for the 13 query-count conditions; do not point it at this directory until its LOQOA support is implemented. Saved predictions can be scored later without loading BLIP-2.
